# Mean Reversion Analysis: Half-Life & Hurst Exponent

This notebook calculates the half-life of mean reversion and Hurst exponent for all trading products, with diagnostics for understanding why calculations may fail for some products.

In [6]:
import pandas as pd
import numpy as np
from scipy.stats import linregress
import glob

# Load data from all 3 days
prices = []
for file in sorted(glob.glob('prices_round_4_day_*.csv')):
    df = pd.read_csv(file, sep=';')
    prices.append(df)

df_total = pd.concat(prices, ignore_index=True)
print(f"Loaded {len(df_total)} price records from {len(prices)} days")
print(f"Unique products: {df_total['product'].nunique()}")

Loaded 360000 price records from 3 days
Unique products: 12


In [7]:
# Calculate Half-Life and Hurst Exponent - WORKING VERSION
# Uses scipy.stats.linregress for robust regression

print("\n" + "="*80)
print("MEAN REVERSION ANALYSIS: Half-Life & Hurst Exponent")
print("="*80)

results_list = []

for product in sorted(df_total['product'].unique()):
    prices_arr = df_total[df_total['product'] == product]['mid_price'].dropna().values
    
    if len(prices_arr) < 50:
        print(f"\n{product}: SKIPPED (only {len(prices_arr)} data points)")
        continue
    
    print(f"\n{product}:")
    
    halflife_val = np.nan
    hurst_val = np.nan
    
    # ===== HALF-LIFE CALCULATION =====
    mean_price = np.mean(prices_arr)
    deviations = prices_arr - mean_price
    
    # AR(1) model: dev_t = slope * dev_{t-1} + error
    X_lagged = deviations[:-1]  # dev_{t-1}
    y_current = deviations[1:]  # dev_t
    
    if np.std(X_lagged) > 1e-12 and len(X_lagged) > 5:
        try:
            # Use robust scipy linregress instead of numpy polyfit
            slope, intercept, r_value, p_value, std_err = linregress(X_lagged, y_current)
            
            if 0 < slope < 1:
                halflife_val = np.log(0.5) / np.log(slope)
                print(f"  ✓ Half-Life: {halflife_val:.2f} periods (AR(1)={slope:.4f})")
            else:
                print(f"  ✗ Half-Life: FAILED - AR(1) slope {slope:.4f} not in (0,1) [not mean-reverting]")
        except Exception as e:
            print(f"  ✗ Half-Life: ERROR - {str(e)[:50]}")
    else:
        print(f"  ✗ Half-Life: FAILED - Insufficient variance or data")
    
    # ===== HURST EXPONENT CALCULATION =====
    try:
        lags_test = np.array([10, 20, 50], dtype=int)
        taus_list = []
        valid_lags = []
        
        for lag in lags_test:
            if lag >= len(prices_arr):
                continue
            mean_val = np.mean(prices_arr[:lag])
            Y_cum = np.cumsum(prices_arr[:lag] - mean_val)
            R = np.max(Y_cum) - np.min(Y_cum)
            S = np.std(prices_arr[:lag])
            
            if S > 0:
                taus_list.append(R / S)
                valid_lags.append(lag)
        
        if len(taus_list) >= 2:
            hurst_val, _, _, _, _ = linregress(np.log(valid_lags), np.log(taus_list))
            classification = "Mean-Reverting" if hurst_val < 0.45 else "Random Walk" if hurst_val < 0.55 else "Trending"
            print(f"  ✓ Hurst: {hurst_val:.3f} ({classification})")
    except Exception as e:
        print(f"  ✗ Hurst: ERROR - {str(e)[:50]}")
    
    results_list.append({
        'Product': product,
        'Half-Life': f"{halflife_val:.2f}" if not np.isnan(halflife_val) else "—",
        'Hurst': f"{hurst_val:.3f}" if not np.isnan(hurst_val) else "—",
        'Data Points': len(prices_arr)
    })

# Display results
print("\n" + "="*80)
print("SUMMARY TABLE")
print("="*80)
results_df = pd.DataFrame(results_list)
print(results_df.to_string(index=False))


MEAN REVERSION ANALYSIS: Half-Life & Hurst Exponent

HYDROGEL_PACK:
  ✓ Half-Life: 349.67 periods (AR(1)=0.9980)
  ✓ Hurst: 0.794 (Trending)

VELVETFRUIT_EXTRACT:
  ✓ Half-Life: 351.90 periods (AR(1)=0.9980)
  ✓ Hurst: 0.862 (Trending)

VEV_4000:
  ✓ Half-Life: 215.95 periods (AR(1)=0.9968)
  ✓ Hurst: 0.858 (Trending)

VEV_4500:
  ✓ Half-Life: 279.46 periods (AR(1)=0.9975)
  ✓ Hurst: 0.899 (Trending)

VEV_5000:
  ✓ Half-Life: 430.39 periods (AR(1)=0.9984)
  ✓ Hurst: 0.916 (Trending)

VEV_5100:
  ✓ Half-Life: 479.56 periods (AR(1)=0.9986)
  ✓ Hurst: 0.910 (Trending)

VEV_5200:
  ✓ Half-Life: 545.33 periods (AR(1)=0.9987)
  ✓ Hurst: 0.918 (Trending)

VEV_5300:
  ✓ Half-Life: 520.81 periods (AR(1)=0.9987)
  ✓ Hurst: 0.817 (Trending)

VEV_5400:
  ✓ Half-Life: 377.03 periods (AR(1)=0.9982)
  ✓ Hurst: 0.849 (Trending)

VEV_5500:
  ✓ Half-Life: 268.22 periods (AR(1)=0.9974)
  ✓ Hurst: 0.876 (Trending)

VEV_6000:
  ✗ Half-Life: FAILED - Insufficient variance or data

VEV_6500:
  ✗ Half-Life: 

In [8]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.filters.hp_filter import hpfilter

for product in sorted(df_total['product'].unique()):
    prices_arr = df_total[df_total['product'] == product]['mid_price'].dropna().values

    # --- Method 1: HP Filter detrending ---
    # Lambda controls smoothness of trend (higher = smoother)
    # 1600 is standard for quarterly, for tick data try much higher
    cycle, trend = hpfilter(prices_arr, lamb=1e6)
    residuals = cycle  # this is P_t - μ_t

    # --- Method 2: Rolling mean detrending ---
    window = 500
    rolling_mean = pd.Series(prices_arr).rolling(window, min_periods=1, center=True).mean().values
    residuals_rolling = prices_arr - rolling_mean

    # --- Hurst on RESIDUALS (not raw price) ---
    def hurst_rs(series):
        lags = np.unique(np.logspace(1, np.log10(len(series)//4), 30).astype(int))
        taus = []
        valid_lags = []
        for lag in lags:
            sub = series[:lag]
            S = np.std(sub)
            if S < 1e-10: continue
            Y_cum = np.cumsum(sub - np.mean(sub))
            R = np.max(Y_cum) - np.min(Y_cum)
            taus.append(R / S)
            valid_lags.append(lag)
        if len(taus) < 3: return np.nan
        h, *_ = linregress(np.log(valid_lags), np.log(taus))
        return h

    h_raw      = hurst_rs(prices_arr)
    h_residual = hurst_rs(residuals)

    # --- ADF test on residuals (more powerful than Hurst) ---
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(residuals, maxlag=20, autolag='AIC')
    
    # --- Half-life on residuals (not global mean) ---
    dev = residuals[1:]
    dev_lag = residuals[:-1]
    slope, *_ = linregress(dev_lag, dev)
    halflife = np.log(0.5) / np.log(abs(slope)) if 0 < slope < 1 else np.nan

    print(f"\n{product}:")
    print(f"  Hurst (raw):      {h_raw:.3f}  ← contaminated by drift")
    print(f"  Hurst (residual): {h_residual:.3f}  ← true mean-reversion signal")
    print(f"  ADF p-value:      {adf_p:.4f}  {'✓ stationary' if adf_p < 0.05 else '✗ non-stationary'}")
    print(f"  Half-life:        {halflife:.1f} periods")


HYDROGEL_PACK:
  Hurst (raw):      0.993  ← contaminated by drift
  Hurst (residual): 0.496  ← true mean-reversion signal
  ADF p-value:      0.0000  ✓ stationary
  Half-life:        8.7 periods

VELVETFRUIT_EXTRACT:
  Hurst (raw):      0.956  ← contaminated by drift
  Hurst (residual): 0.510  ← true mean-reversion signal
  ADF p-value:      0.0000  ✓ stationary
  Half-life:        8.3 periods

VEV_4000:
  Hurst (raw):      0.956  ← contaminated by drift
  Hurst (residual): 0.508  ← true mean-reversion signal
  ADF p-value:      0.0000  ✓ stationary
  Half-life:        5.2 periods

VEV_4500:
  Hurst (raw):      0.957  ← contaminated by drift
  Hurst (residual): 0.511  ← true mean-reversion signal
  ADF p-value:      0.0000  ✓ stationary
  Half-life:        6.6 periods

VEV_5000:
  Hurst (raw):      0.956  ← contaminated by drift
  Hurst (residual): 0.511  ← true mean-reversion signal
  ADF p-value:      0.0000  ✓ stationary
  Half-life:        9.7 periods

VEV_5100:
  Hurst (raw):    